# Language-Aware Layer Pruning in Small Multilingual LLMs

**Research title:** Do Small Multilingual LLMs Share Redundant Layers Across Languages?  
**Languages:** English, Hindi, Marathi  
**Primary model:** Qwen2.5-1.5B-Instruct  
**Replication models:** Llama-3.2-1B, Gemma-2-2B

## Goal

Test whether Transformer layers that look redundant on English calibration text are also redundant for Hindi and Marathi.

## Core idea

For every Transformer block:

`Block Influence = 1 - cosine(layer_input, layer_output)`

Low influence means the block changes its hidden representation less on the calibration text. This is only a **proxy for redundancy**, not proof that the layer is unnecessary.

We then actually remove the lowest-ranked layers and measure perplexity on all three languages.

## Research questions

- **RQ1:** Do layer-importance rankings differ across English, Hindi, and Marathi?
- **RQ2:** Does English-calibrated pruning hurt Indic perplexity more than language-matched pruning?
- **RQ3:** Can a balanced English+Hindi+Marathi calibration set reduce this mismatch?
- **RQ4:** Are ranking differences related to tokenizer token-per-word behavior?

## Experimental conditions

Calibration:
1. English
2. Hindi
3. Marathi
4. Balanced English+Hindi+Marathi

Pruning:
- 2 layers
- 4 layers
- 6 layers

Evaluation:
- English
- Hindi
- Marathi

## Important rule

Run the notebook **top to bottom**. Start with Qwen. Only run Llama/Gemma after the Qwen pipeline works.

## Step 0: Colab setup

1. Open Google Colab.
2. Select a GPU runtime.
3. Upload this notebook or open it from the downloaded file.
4. Run cells from top to bottom.
5. The first model to run is Qwen2.5-1.5B-Instruct.
6. Save the generated CSV/PNG files.
7. Send the results back for interpretation and paper writing.

The notebook uses streaming Wikipedia data, so it does not download the complete Wikipedia corpus.

In [ ]:
!pip -q install -U transformers datasets accelerate sentencepiece scipy pandas matplotlib seaborn

## Step 1: Imports and configuration

Change only the values in the configuration cell if Colab memory is limited.

Start with:
- `MODEL_KEY = "qwen"`
- `CALIBRATION_CHUNKS_PER_LANGUAGE = 20`
- `EVAL_CHUNKS_PER_LANGUAGE = 20`

For a final paper, increase the sample size and repeat important experiments with multiple seeds.

In [ ]:
import gc
import math
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from datasets import load_dataset
from scipy.stats import spearmanr, kendalltau
from transformers import AutoTokenizer, AutoModelForCausalLM

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_KEY = "qwen"

MODELS = {
    "qwen": "Qwen/Qwen2.5-1.5B-Instruct",
    "llama": "meta-llama/Llama-3.2-1B",
    "gemma": "google/gemma-2-2b",
}

LANG_CODES = {
    "en": "20231101.en",
    "hi": "20231101.hi",
    "mr": "20231101.mr",
}

CALIBRATION_CHUNKS_PER_LANGUAGE = 20
EVAL_CHUNKS_PER_LANGUAGE = 20

SEQ_LEN = 256
BATCH_SIZE = 1
MAX_NEW_DATASET_ROWS = 2000

PRUNE_LEVELS = [2, 4, 6]

OUTPUT_DIR = Path("/content/layer_pruning_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else (
    torch.float16 if torch.cuda.is_available() else torch.float32
)

print("Device:", DEVICE)
print("Dtype:", DTYPE)
print("Model:", MODELS[MODEL_KEY])

## Step 2: Load tokenizer and model

The loader supports the three target model families.

If a model requires authentication, log in to Hugging Face in Colab first. Do not put access tokens directly into the notebook.

In [ ]:
def load_model(model_key):
    model_name = MODELS[model_key]

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    kwargs = dict(
        torch_dtype=DTYPE,
        device_map="auto" if torch.cuda.is_available() else None,
        trust_remote_code=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        **kwargs
    )

    if not torch.cuda.is_available():
        model = model.to(DEVICE)

    model.eval()
    return tokenizer, model

tokenizer, model = load_model(MODEL_KEY)

print("Loaded:", MODELS[MODEL_KEY])
print("Tokenizer vocab size:", len(tokenizer))

## Step 3: Find Transformer blocks

Different model families store their Transformer blocks under different attributes. This helper finds the correct block list.

We will not change the model yet.

In [ ]:
def get_transformer_blocks(model):
    candidates = [
        ("model.layers", lambda m: m.model.layers),
        ("model.decoder.layers", lambda m: m.model.decoder.layers),
        ("transformer.h", lambda m: m.transformer.h),
        ("decoder.layers", lambda m: m.decoder.layers),
    ]

    for name, getter in candidates:
        try:
            blocks = getter(model)
            if blocks is not None and len(blocks) > 0:
                return blocks, name
        except Exception:
            pass

    raise RuntimeError("Could not find Transformer blocks.")

blocks, block_path = get_transformer_blocks(model)

print("Block path:", block_path)
print("Number of layers:", len(blocks))

## Step 4: Load multilingual calibration and evaluation text

We use Wikimedia Wikipedia subsets for English, Hindi, and Marathi.

**Calibration text** is used only to rank layers.

**Evaluation text** is kept separate and is used to calculate perplexity.

Do not mix calibration and evaluation examples in the final experiment.

In [ ]:
def load_wikipedia_text(lang_code, max_rows=2000):
    ds = load_dataset(
        "wikimedia/wikipedia",
        lang_code,
        split="train",
        streaming=True
    )

    texts = []
    for row in ds:
        text = row.get("text", "")
        if text and len(text.split()) >= 30:
            texts.append(text)
        if len(texts) >= max_rows:
            break

    return texts


def build_chunks(texts, tokenizer, seq_len=256, max_chunks=20):
    chunks = []

    for text in texts:
        ids = tokenizer(
            text,
            add_special_tokens=False,
            truncation=False
        )["input_ids"]

        for start in range(0, len(ids) - seq_len + 1, seq_len):
            chunk = ids[start:start + seq_len]
            if len(chunk) == seq_len:
                chunks.append(torch.tensor(chunk, dtype=torch.long))
            if len(chunks) >= max_chunks:
                return chunks

    return chunks


def load_language_chunks(lang, calibration=True):
    code_name = LANG_CODES[lang]
    texts = load_wikipedia_text(code_name, MAX_NEW_DATASET_ROWS)

    n = CALIBRATION_CHUNKS_PER_LANGUAGE if calibration else EVAL_CHUNKS_PER_LANGUAGE
    return build_chunks(texts, tokenizer, SEQ_LEN, n)


calibration_chunks = {}
eval_chunks = {}

for lang in ["en", "hi", "mr"]:
    print("Loading", lang)
    calibration_chunks[lang] = load_language_chunks(lang, calibration=True)
    eval_chunks[lang] = load_language_chunks(lang, calibration=False)
    print(
        lang,
        "calibration:", len(calibration_chunks[lang]),
        "evaluation:", len(eval_chunks[lang])
    )

## Step 5: Check the data

You need enough fixed-length chunks for every language.

For the pilot experiment, 20 chunks per language is enough to test the pipeline.

For a paper, use a substantially larger and independently sampled calibration/evaluation set.

In [ ]:
for lang in ["en", "hi", "mr"]:
    print(
        lang,
        "calibration chunks =", len(calibration_chunks[lang]),
        "| evaluation chunks =", len(eval_chunks[lang])
    )

## Step 6: Measure Block Influence

For each Transformer block:

1. Save its input hidden states.
2. Get its output hidden states.
3. Compute cosine similarity between input and output.
4. Convert it to influence:

`Influence = 1 - cosine_similarity`

Average over tokens and calibration samples.

**Interpretation:** lower influence = smaller average representation change.

Important: this is a ranking signal, not a causal proof of importance.

In [ ]:
class BlockInputCapture:
    def __init__(self):
        self.input_states = {}

    def hook(self, idx):
        def fn(module, inputs):
            if not inputs:
                return
            x = inputs[0]
            if torch.is_tensor(x):
                self.input_states[idx] = x.detach()
        return fn


@torch.no_grad()
def measure_block_influence(model, chunks, batch_size=1):
    blocks, _ = get_transformer_blocks(model)
    capture = BlockInputCapture()

    handles = []
    for i, block in enumerate(blocks):
        handles.append(block.register_forward_pre_hook(capture.hook(i)))

    sums = np.zeros(len(blocks), dtype=np.float64)
    counts = np.zeros(len(blocks), dtype=np.int64)

    try:
        for start in range(0, len(chunks), batch_size):
            batch = chunks[start:start + batch_size]
            input_ids = torch.stack(batch).to(model.device)

            capture.input_states.clear()

            outputs = model(
                input_ids=input_ids,
                use_cache=False
            )

            for i, block in enumerate(blocks):
                x = capture.input_states.get(i)
                y = outputs.hidden_states[i + 1] if getattr(outputs, "hidden_states", None) is not None else None

                # The model must expose hidden states for the fallback-free path.
                # We therefore re-run below with output_hidden_states=True.
                if y is None:
                    raise RuntimeError("Hidden states were not returned.")

                x = x.float().reshape(-1, x.shape[-1])
                y = y.float().reshape(-1, y.shape[-1])

                cosine = F.cosine_similarity(x, y, dim=-1)
                influence = (1.0 - cosine).mean().item()

                sums[i] += influence
                counts[i] += 1

    finally:
        for h in handles:
            h.remove()

    return sums / np.maximum(counts, 1)


# Correct implementation using output_hidden_states=True
@torch.no_grad()
def measure_block_influence(model, chunks, batch_size=1):
    blocks, _ = get_transformer_blocks(model)
    capture = BlockInputCapture()

    handles = [
        block.register_forward_pre_hook(capture.hook(i))
        for i, block in enumerate(blocks)
    ]

    sums = np.zeros(len(blocks), dtype=np.float64)
    counts = np.zeros(len(blocks), dtype=np.int64)

    try:
        for start in range(0, len(chunks), batch_size):
            batch = chunks[start:start + batch_size]
            input_ids = torch.stack(batch).to(model.device)
            capture.input_states.clear()

            outputs = model(
                input_ids=input_ids,
                use_cache=False,
                output_hidden_states=True
            )

            for i in range(len(blocks)):
                x = capture.input_states[i]
                y = outputs.hidden_states[i + 1]

                x = x.float().reshape(-1, x.shape[-1])
                y = y.float().reshape(-1, y.shape[-1])

                cosine = F.cosine_similarity(x, y, dim=-1)
                influence = (1.0 - cosine).mean().item()

                sums[i] += influence
                counts[i] += 1

    finally:
        for h in handles:
            h.remove()

    return sums / np.maximum(counts, 1)

## Step 7: Calculate rankings for English, Hindi, and Marathi

Run the influence measurement separately for each language.

The output gives one influence value per Transformer layer.

In [ ]:
influence_by_lang = {}

for lang in ["en", "hi", "mr"]:
    print("Measuring:", lang)
    influence_by_lang[lang] = measure_block_influence(
        model,
        calibration_chunks[lang],
        batch_size=BATCH_SIZE
    )

influence_df = pd.DataFrame(influence_by_lang)
influence_df.index.name = "layer"

display(influence_df)

## Step 8: Rank layers and compare languages

A layer with low influence receives a low rank and becomes a pruning candidate.

We compare rankings using:
- Spearman correlation
- Kendall tau

Higher correlation means the two languages produce more similar rankings.

In [ ]:
rank_df = influence_df.rank(method="average", ascending=True)

pairs = [("en", "hi"), ("en", "mr"), ("hi", "mr")]
rank_results = []

for a, b in pairs:
    rho, rho_p = spearmanr(rank_df[a], rank_df[b])
    tau, tau_p = kendalltau(rank_df[a], rank_df[b])

    rank_results.append({
        "language_a": a,
        "language_b": b,
        "spearman_rho": rho,
        "spearman_p": rho_p,
        "kendall_tau": tau,
        "kendall_p": tau_p
    })

rank_results_df = pd.DataFrame(rank_results)
display(rank_results_df)

influence_df.to_csv(OUTPUT_DIR / "block_influence_by_language.csv")
rank_results_df.to_csv(OUTPUT_DIR / "rank_agreement.csv", index=False)

## Step 9: Plot layer influence

This plot is one of the main diagnostic figures.

Look for:
- layers that are consistently low across languages
- layers that are low in English but high in Hindi/Marathi
- different curves or ranking patterns

In [ ]:
plt.figure(figsize=(10, 5))

for lang in ["en", "hi", "mr"]:
    plt.plot(
        influence_df.index,
        influence_df[lang],
        marker="o",
        label=lang
    )

plt.xlabel("Transformer layer")
plt.ylabel("Block Influence (1 - cosine similarity)")
plt.title("Layer Influence Across Languages")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "layer_influence.png", dpi=200)
plt.show()

## Step 10: Build balanced mixed-language calibration

The mixed condition contains equal numbers of English, Hindi, and Marathi chunks.

This tests whether a small multilingual calibration set can produce a more language-robust pruning ranking.

In [ ]:
def balanced_mix(chunks_by_lang):
    n = min(len(v) for v in chunks_by_lang.values())

    mixed = []
    for i in range(n):
        for lang in ["en", "hi", "mr"]:
            mixed.append(chunks_by_lang[lang][i])

    return mixed


mixed_calibration = balanced_mix(calibration_chunks)

print("Mixed calibration chunks:", len(mixed_calibration))

influence_by_lang["mixed"] = measure_block_influence(
    model,
    mixed_calibration,
    batch_size=BATCH_SIZE
)

influence_df = pd.DataFrame(influence_by_lang)
influence_df.index.name = "layer"

display(influence_df)

## Step 11: Identify layers to prune

For every calibration condition, select the lowest-influence layers.

Example:
- `k=2`: remove 2 lowest-influence layers
- `k=4`: remove 4
- `k=6`: remove 6

Never modify the original model permanently. For every pruning experiment, reload a fresh copy of the model.

In [ ]:
def lowest_influence_layers(influence_values, k):
    return list(np.argsort(influence_values)[:k])


pruning_sets = {}

for condition in ["en", "hi", "mr", "mixed"]:
    pruning_sets[condition] = {}

    for k in PRUNE_LEVELS:
        layers_to_remove = lowest_influence_layers(
            influence_df[condition].values,
            k
        )
        pruning_sets[condition][k] = layers_to_remove

print(json.dumps(pruning_sets, indent=2))

## Step 12: Prune a fresh model

The function removes complete Transformer blocks.

Because architectures store their blocks differently, the helper uses the block path discovered earlier.

**Important:** pruning by deleting layers changes the model architecture. This is an experimental structural-pruning method, not merely skipping computation.

In [ ]:
def set_transformer_blocks(model, new_blocks, block_path):
    if block_path == "model.layers":
        model.model.layers = nn.ModuleList(new_blocks)
    elif block_path == "model.decoder.layers":
        model.model.decoder.layers = nn.ModuleList(new_blocks)
    elif block_path == "transformer.h":
        model.transformer.h = nn.ModuleList(new_blocks)
    elif block_path == "decoder.layers":
        model.decoder.layers = nn.ModuleList(new_blocks)
    else:
        raise RuntimeError("Unknown block path.")


def prune_model(model, layer_indices):
    blocks, path = get_transformer_blocks(model)

    remove = set(layer_indices)
    kept = [
        block for i, block in enumerate(blocks)
        if i not in remove
    ]

    set_transformer_blocks(model, kept, path)
    return model

## Step 13: Perplexity evaluation

Perplexity is calculated on held-out text.

Lower perplexity means the model assigns higher probability to the evaluation text.

Compare every pruned model against the unpruned baseline.

Relative degradation:

`100 × (PPL_pruned - PPL_baseline) / PPL_baseline`

In [ ]:
@torch.no_grad()
def perplexity(model, chunks):
    model.eval()

    total_nll = 0.0
    total_tokens = 0

    for chunk in chunks:
        input_ids = chunk.unsqueeze(0).to(model.device)

        outputs = model(
            input_ids=input_ids,
            labels=input_ids,
            use_cache=False
        )

        # HF loss is mean token NLL for the shifted sequence.
        token_count = input_ids.shape[1] - 1

        total_nll += outputs.loss.item() * token_count
        total_tokens += token_count

    return math.exp(total_nll / total_tokens)


baseline_ppl = {}

for lang in ["en", "hi", "mr"]:
    baseline_ppl[lang] = perplexity(model, eval_chunks[lang])

print("Baseline PPL:", baseline_ppl)

## Step 14: Run the full pruning sweep

This is the main experiment.

For every calibration condition:
- choose 2/4/6 lowest-influence layers
- reload the original model
- prune those layers
- evaluate PPL on English, Hindi, Marathi

This produces the main cross-language degradation table.

In [ ]:
results = []

for condition in ["en", "hi", "mr", "mixed"]:
    for k in PRUNE_LEVELS:
        layers_to_remove = pruning_sets[condition][k]

        print(
            f"Condition={condition}, remove={k}, layers={layers_to_remove}"
        )

        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        tokenizer, fresh_model = load_model(MODEL_KEY)
        prune_model(fresh_model, layers_to_remove)

        for eval_lang in ["en", "hi", "mr"]:
            ppl = perplexity(fresh_model, eval_chunks[eval_lang])
            base = baseline_ppl[eval_lang]
            degradation = 100 * (ppl - base) / base

            results.append({
                "model": MODEL_KEY,
                "calibration": condition,
                "pruned_layers": k,
                "removed_layer_ids": str(layers_to_remove),
                "evaluation_language": eval_lang,
                "baseline_ppl": base,
                "pruned_ppl": ppl,
                "relative_ppl_degradation_pct": degradation
            })

        del fresh_model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

results_df = pd.DataFrame(results)
results_df.to_csv(
    OUTPUT_DIR / "pruning_perplexity_results.csv",
    index=False
)

display(results_df)

## Step 15: Plot perplexity degradation

The most important comparison is:

**English-calibrated pruning vs Hindi/Marathi evaluation**

If English ranking selects layers that are less suitable for Indic text, Indic degradation should be larger than degradation under language-matched calibration.

Do not claim this from the plot alone. Check the actual numbers and uncertainty.

In [ ]:
for eval_lang in ["en", "hi", "mr"]:
    subset = results_df[
        results_df["evaluation_language"] == eval_lang
    ]

    pivot = subset.pivot(
        index="pruned_layers",
        columns="calibration",
        values="relative_ppl_degradation_pct"
    )

    pivot.plot(
        marker="o",
        figsize=(10, 5)
    )

    plt.axhline(0, linewidth=1)
    plt.title(f"Relative PPL degradation on {eval_lang}")
    plt.xlabel("Number of removed layers")
    plt.ylabel("Relative PPL degradation (%)")
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(
        OUTPUT_DIR / f"ppl_degradation_{eval_lang}.png",
        dpi=200
    )
    plt.show()

## Step 16: Tokenizer analysis

Measure tokens per whitespace-separated word.

This is a descriptive analysis. It does not prove that tokenization causes pruning differences.

Compare:
- English
- Hindi
- Marathi

Use the same text sampling procedure for all languages.

In [ ]:
def tokens_per_word(texts, tokenizer, max_texts=100):
    values = []

    for text in texts[:max_texts]:
        words = text.split()
        if not words:
            continue

        token_ids = tokenizer(
            text,
            add_special_tokens=False
        )["input_ids"]

        values.append(len(token_ids) / len(words))

    return float(np.mean(values))


token_stats = []

raw_texts = {}

for lang in ["en", "hi", "mr"]:
    raw_texts[lang] = load_wikipedia_text(
        LANG_CODES[lang],
        max_rows=100
    )

    tpw = tokens_per_word(
        raw_texts[lang],
        tokenizer
    )

    token_stats.append({
        "language": lang,
        "tokens_per_word": tpw
    })

token_stats_df = pd.DataFrame(token_stats)
display(token_stats_df)

token_stats_df.to_csv(
    OUTPUT_DIR / "tokenizer_tokens_per_word.csv",
    index=False
)

## Step 17: Ranking stability

A good pruning ranking should not change dramatically when the calibration sample changes.

We calculate Jaccard similarity between the top-k candidate sets from two calibration samples.

For a final paper, repeat this over several random calibration seeds and report mean ± standard deviation.

In [ ]:
def jaccard(a, b):
    a, b = set(a), set(b)
    if not a and not b:
        return 1.0
    return len(a & b) / len(a | b)


def shuffled_split(chunks, seed):
    rng = random.Random(seed)
    items = chunks.copy()
    rng.shuffle(items)
    return items


stability_rows = []

for lang in ["en", "hi", "mr"]:
    split_a = shuffled_split(calibration_chunks[lang], 1)
    split_b = shuffled_split(calibration_chunks[lang], 2)

    inf_a = measure_block_influence(model, split_a, BATCH_SIZE)
    inf_b = measure_block_influence(model, split_b, BATCH_SIZE)

    for k in PRUNE_LEVELS:
        a = lowest_influence_layers(inf_a, k)
        b = lowest_influence_layers(inf_b, k)

        stability_rows.append({
            "language": lang,
            "k": k,
            "jaccard": jaccard(a, b)
        })

stability_df = pd.DataFrame(stability_rows)
display(stability_df)

stability_df.to_csv(
    OUTPUT_DIR / "ranking_stability.csv",
    index=False
)

## Step 18: Optional random-pruning baseline

This control is important for publication.

If influence-based pruning is not better than random pruning, the proposed ranking method is not demonstrating useful selection.

Use multiple random seeds for a stronger final experiment.

In [ ]:
def random_pruning_sets(num_layers, k, seeds=(1, 2, 3)):
    output = []

    for seed in seeds:
        rng = random.Random(seed)
        ids = list(range(num_layers))
        rng.shuffle(ids)

        output.append({
            "seed": seed,
            "k": k,
            "layers": sorted(ids[:k])
        })

    return output


num_layers = len(influence_df)

for k in PRUNE_LEVELS:
    print("Random baseline k =", k)
    print(random_pruning_sets(num_layers, k))

## Step 19: Save a single experiment summary

After the notebook finishes, download or keep the entire output directory.

Important files:
- `block_influence_by_language.csv`
- `rank_agreement.csv`
- `pruning_perplexity_results.csv`
- `tokenizer_tokens_per_word.csv`
- `ranking_stability.csv`
- PNG plots

These are the files to send back for analysis.

In [ ]:
summary = {
    "model": MODEL_KEY,
    "model_name": MODELS[MODEL_KEY],
    "languages": ["en", "hi", "mr"],
    "calibration_conditions": ["en", "hi", "mr", "mixed"],
    "prune_levels": PRUNE_LEVELS,
    "seq_len": SEQ_LEN,
    "calibration_chunks_per_language": CALIBRATION_CHUNKS_PER_LANGUAGE,
    "evaluation_chunks_per_language": EVAL_CHUNKS_PER_LANGUAGE,
    "seed": SEED,
}

with open(OUTPUT_DIR / "experiment_config.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Saved results to:", OUTPUT_DIR)
print("Files:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(" -", p.name)

# What to send back

After running the notebook, send:

1. `block_influence_by_language.csv`
2. `rank_agreement.csv`
3. `pruning_perplexity_results.csv`
4. `tokenizer_tokens_per_word.csv`
5. `ranking_stability.csv`
6. The generated PNG plots
7. Any error messages if a model failed

Then the results can be converted into:
- Results section
- statistical analysis
- tables and figures
- discussion
- limitations
- conclusion
- IEEE-style paper wording

## Final-paper warning

Do not report the pilot numbers as final scientific evidence. Increase calibration/evaluation samples, add random baselines, use multiple seeds, and document exact model/tokenizer versions before publication.